In [1]:
# This is necessary to recognize the modules
import os
import sys
from decimal import Decimal

root_path = os.path.abspath(os.path.join(os.getcwd(), '../..'))
sys.path.append(root_path)

In [2]:
from data_sources.clob import CLOBDataSource

# Get trading rules and candles
clob = CLOBDataSource()

In [3]:
# Constants
CONNECTOR_NAME = "binance_perpetual"
INTERVALS = ["1m"]
DAYS = 15
FETCH_CANDLES = False


# Download data
- Get trading rules
- Get candles for the last x days

In [5]:
import asyncio

trading_rules = await clob.get_trading_rules(CONNECTOR_NAME)
trading_pairs = trading_rules.filter_by_quote_asset("USDT")\
    .filter_by_min_notional_size(Decimal("5"))\
    .get_all_trading_pairs()

In [6]:
BATCH_CANDLES_REQUEST = 15
SLEEP_REQUEST = 60.0


if FETCH_CANDLES:
    number_of_calls = (len(trading_pairs) // BATCH_CANDLES_REQUEST) + 1

    all_candles = {}

    for i in range(number_of_calls):
        print(f"Batch {i + 1}/{number_of_calls}")
        start = i * BATCH_CANDLES_REQUEST
        end = (i + 1) * BATCH_CANDLES_REQUEST
        print(f"Start: {start}, End: {end}")
        end = min(end, len(trading_pairs))
        trading_pairs_batch = trading_pairs[start:end]

        tasks = [clob.get_candles_last_days(
        connector_name=CONNECTOR_NAME,
        trading_pair=trading_pair,
        interval=interval,
        days=DAYS,
        ) for trading_pair in trading_pairs_batch for interval in INTERVALS]

        candles = await asyncio.gather(*tasks)
        candles = {trading_pair: candle for trading_pair, candle in zip(trading_pairs, candles)}
        all_candles.update(candles)
        if i != number_of_calls - 1:
            print(f"Sleeping for {SLEEP_REQUEST} seconds")
            await asyncio.sleep(SLEEP_REQUEST)
    clob.dump_candles_cache(os.path.join(root_path, "data"))
else:
    clob.load_candles_cache(os.path.join(root_path, "data"))

In [13]:
candles_1m = [value for key, value in clob.candles_cache.items() if key[2] == "1m"]

In [15]:
from research_notebooks.xtreet_bb.utils import generate_report

# Features configuration
BOLLINGER_LENGTH = 50
BOLLINGER_STD = 1.0

report = generate_report(
    candles=candles_1m,
    BOLLINGER_LENGTH=BOLLINGER_LENGTH,
    BOLLINGER_STD=BOLLINGER_STD,
)
report

,trading_pair,worst_q0,worst_q1,worst_q2,worst_q3,worst_q4,max_rev_q0,max_rev_q1,max_rev_q2,max_rev_q3,max_rev_q4,n_right_reversions,n_fake_reversions,street_cross,n_out_of_bounds,risk_ration_mean
0,PIXEL-USDT,0,0.00212465,0.004947,0.01184834,0.17615793,0.00071633,0.00373413,0.00555115,0.00791367,0.02407407,375,121,509,1732,2.15867033
1,NOT-USDT,0,0.00238495,0.00607269,0.0143222,0.15698314,0.00060303,0.00352463,0.00558313,0.00833856,0.03287079,332,141,473,1668,3.09674679
2,KEY-USDT,0,0.00140442,0.00415781,0.0095017,0.18465977,0,0.0030388,0.00473678,0.00686903,0.02762431,387,125,514,1735,3.12107056
3,LSK-USDT,0,0.00127952,0.0039556,0.01033167,0.1259111,0.00034184,0.002659,0.00388889,0.00602365,0.01965318,355,134,491,1713,2.98305075
4,ENJ-USDT,0,0.00117123,0.00333406,0.00865805,0.1548037,0.00028339,0.00270507,0.00407424,0.00588165,0.02733376,408,127,535,1685,4.39204906
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
256,T-USDT,0,0.0016488,0.00437397,0.01181335,0.26182965,0.00049237,0.003,0.00452716,0.00683761,0.0268714,344,122,473,1671,2.44078405
257,COMP-USDT,0,0.00174,0.0040937,0.0096516,0.15394089,0.00045819,0.00260787,0.00416378,0.00635076,0.0221791,356,135,493,1764,3.01001852
258,ONDO-USDT,0,0.00204668,0.00620424,0.01441845,0.39741887,0.00039641,0.00337529,0.0053569,0.00805944,0.03167775,316,138,459,1652,3.77033314
259,IMX-USDT,0,0.00142809,0.00399055,0.0097561,0.16528247,0.00026199,0.00273755,0.00435092,0.00659359,0.0267866,364,135,501,1691,3.89850443


In [16]:
report["right_fake_ratio"] = report["n_right_reversions"] / report["n_fake_reversions"]

report.sort_values("right_fake_ratio", ascending=False)

,trading_pair,worst_q0,worst_q1,worst_q2,worst_q3,worst_q4,max_rev_q0,max_rev_q1,max_rev_q2,max_rev_q3,max_rev_q4,n_right_reversions,n_fake_reversions,street_cross,n_out_of_bounds,risk_ration_mean,right_fake_ratio
13,XEM-USDT,0,0,0,0.00903986,0.19123506,0.00398406,0.00877193,0.00952381,0.01304348,0.03529412,569,80,764,3249,1.19780424,7.1125
107,CRV-USDT,0,0,0.00390625,0.01045296,0.17813765,0.00311526,0.00772201,0.00809717,0.0106383,0.02816901,510,89,662,3051,1.20386304,5.73033708
157,CELO-USDT,0,0,0.00224467,0.00668151,0.16778523,0,0.00421053,0.0043956,0.00659708,0.02777778,470,83,614,2494,1.17463309,5.6626506
92,EOS-USDT,0,0,0.00211416,0.00628273,0.14712154,0.00192678,0.00406092,0.0043573,0.00651467,0.02156863,493,96,639,2733,1.32034585,5.13541667
22,LINA-USDT,0,0,0.00249377,0.00753769,0.2309417,0,0.00473934,0.00582524,0.00770713,0.03154574,481,94,625,2655,1.32578703,5.11702128
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
97,TURBO-USDT,0,0.0030581,0.00837696,0.02028844,0.1613723,0.00072692,0.00474547,0.00743018,0.01080369,0.03818616,308,156,465,1636,2.93785217,1.97435897
155,OM-USDT,0,0.00207813,0.0062811,0.01467715,0.12891128,0.00044225,0.00343922,0.00520415,0.00735644,0.05092038,301,153,454,1585,4.56019655,1.96732026
54,1000RATS-USDT,0,0.00268979,0.00793154,0.01764827,0.18882316,0.00097087,0.00443211,0.00700853,0.01012787,0.03053675,301,156,459,1652,3.51807891,1.92948718
185,USDC-USDT,0,0,0.00001501,0.00007406,0.00987326,0,0.00000325,0.00003002,0.00006105,0.00076615,187,119,322,1431,6.40483018,1.57142857


In [17]:
import plotly.express as px


fig = px.scatter(report, x="n_out_of_bounds", y="right_fake_ratio", color="trading_pair")
fig.show()

In [18]:
fig = px.scatter(report, x="n_out_of_bounds", y="max_rev_q3", color="trading_pair")
fig.show()

In [20]:
import plotly.express as px

# Create a scatter matrix (pairplot) using px.scatter_matrix
fig = px.scatter_matrix(
    report,
    dimensions=["n_out_of_bounds", "right_fake_ratio", "max_rev_q1", "max_rev_q2", "worst_q1", "worst_q2"],
    color="trading_pair",  # Optional: color by categorical column
    # hover_data={"hover_info": True},  # Optional: include additional hover info
    title="Pairplot of Selected Features",
    height=2000,
    width=2000,
)

# Show the figure
fig.show()


In [29]:
filtered_report = report[(report["n_out_of_bounds"] > 1700) & (report["right_fake_ratio"] > 3.5)]
filtered_report


,trading_pair,worst_q0,worst_q1,worst_q2,worst_q3,worst_q4,max_rev_q0,max_rev_q1,max_rev_q2,max_rev_q3,max_rev_q4,n_right_reversions,n_fake_reversions,street_cross,n_out_of_bounds,risk_ration_mean,right_fake_ratio
13,XEM-USDT,0,0,0,0.00903986,0.19123506,0.00398406,0.00877193,0.00952381,0.01304348,0.03529412,569,80,764,3249,1.19780424,7.1125
15,LUNA2-USDT,0,0.00118872,0.00370476,0.00894313,0.19486356,0.00030358,0.00277649,0.00447227,0.00684841,0.02931229,411,117,535,1755,3.18534201,3.51282051
17,GALA-USDT,0,0.00175208,0.00426054,0.00961712,0.16895762,0,0.00353357,0.00501788,0.00735809,0.02948886,409,116,536,1772,2.48296361,3.52586207
20,XTZ-USDT,0,0,0.00290698,0.00614441,0.14415781,0,0.00293255,0.00421941,0.00500671,0.02006689,426,96,563,2240,1.30076914,4.4375
22,LINA-USDT,0,0,0.00249377,0.00753769,0.2309417,0,0.00473934,0.00582524,0.00770713,0.03154574,481,94,625,2655,1.32578703,5.11702128
29,BIGTIME-USDT,0,0.00144929,0.00473934,0.01165728,0.21273713,0.00110988,0.00426591,0.00604595,0.0089153,0.02669039,412,110,539,1936,1.88504142,3.74545455
38,ATA-USDT,0,0.00132231,0.00286738,0.0080429,0.16760563,0,0.00271649,0.00416956,0.00586726,0.03059581,397,111,532,2044,1.42141266,3.57657658
43,ONE-USDT,0,0.00096666,0.00370031,0.00909723,0.15538291,0,0.00281492,0.00451671,0.006686,0.0282167,411,108,538,1735,2.0520023,3.80555556
48,FLM-USDT,0,0,0.00209644,0.00758654,0.16242661,0.00169492,0.00373832,0.00518722,0.00615701,0.02702703,431,97,578,2425,1.31082016,4.44329897
56,GTC-USDT,0,0.00150376,0.00326797,0.00956938,0.18499127,0,0.003125,0.00468019,0.00665779,0.02123894,388,107,529,2111,1.43780172,3.62616822


In [31]:
from research_notebooks.xtreet_bb.utils import filter_top_markets

TOP_X_MARKETS = 21  # Number of top markets to select
# MIN_VOLUME_USD = 2000  # Minimum volume in USD
# MIN_NATR = 0.008  # Minimum ATR
# TREND_THRESHOLD = -0.5  # Trend threshold
# TODO: add dca distribution viz
top_markets = filter_top_markets(report_df=filtered_report, top_x=TOP_X_MARKETS)
top_markets

,trading_pair,worst_q0,worst_q1,worst_q2,worst_q3,worst_q4,max_rev_q0,max_rev_q1,max_rev_q2,max_rev_q3,max_rev_q4,n_right_reversions,n_fake_reversions,street_cross,n_out_of_bounds,risk_ration_mean,right_fake_ratio
202,CAKE-USDT,0,0.00107561,0.00287356,0.00667896,0.18529872,0.00029942,0.00220723,0.00332497,0.00521039,0.02283476,432,118,553,1711,3.93120798,3.66101695
15,LUNA2-USDT,0,0.00118872,0.00370476,0.00894313,0.19486356,0.00030358,0.00277649,0.00447227,0.00684841,0.02931229,411,117,535,1755,3.18534201,3.51282051
17,GALA-USDT,0,0.00175208,0.00426054,0.00961712,0.16895762,0,0.00353357,0.00501788,0.00735809,0.02948886,409,116,536,1772,2.48296361,3.52586207
43,ONE-USDT,0,0.00096666,0.00370031,0.00909723,0.15538291,0,0.00281492,0.00451671,0.006686,0.0282167,411,108,538,1735,2.0520023,3.80555556
29,BIGTIME-USDT,0,0.00144929,0.00473934,0.01165728,0.21273713,0.00110988,0.00426591,0.00604595,0.0089153,0.02669039,412,110,539,1936,1.88504142,3.74545455
205,ALGO-USDT,0,0.00086524,0.00322191,0.00756303,0.18596491,0,0.00252048,0.00364797,0.00564251,0.02147239,415,114,552,1856,1.88389379,3.64035088
230,DYDX-USDT,0,0.00103093,0.00366636,0.00852273,0.1627907,0,0.00300601,0.00412371,0.00641026,0.01963048,429,114,561,1947,1.79531433,3.76315789
99,CELR-USDT,0,0.0009403,0.00289995,0.00754189,0.16603416,0,0.00278681,0.00396041,0.00581681,0.03192702,423,105,542,1949,1.76935469,4.02857143
249,LIT-USDT,0,0.0015314,0.00315457,0.00854711,0.14107143,0.00143678,0.00331675,0.00494234,0.00719424,0.03382664,427,104,563,2269,1.60307124,4.10576923
73,OGN-USDT,0,0.00129366,0.0029985,0.00816327,0.14557823,0,0.00312012,0.00436681,0.00669643,0.03109656,420,112,557,2031,1.52827218,3.75


In [34]:
from research_notebooks.xtreet_bb.utils import generate_config, dump_dict_to_yaml


TOTAL_AMOUNT = 3000  # General total amount for all markets
ACTIVATION_BOUNDS = 0.002  # Input activation bounds
MAX_EXECUTORS_PER_SIDE = 1  # Maximum number of executors per side
COOLDOWN_TIME = 0
LEVERAGE = 20 # Should be for each trading pair
TIME_LIMIT = 60 * 60 * 8
BOLLINGER_LENGTHS = [20, 50, 100 ,200]
BOLLINGER_STDS = [1.0]

# DCA amounts
strategy_configs = generate_config(
    intervals=INTERVALS,
    connector_name=CONNECTOR_NAME,
    top_markets=top_markets,
    total_amount=TOTAL_AMOUNT,
    max_executors_per_side=MAX_EXECUTORS_PER_SIDE,
    cooldown_time=COOLDOWN_TIME,
    leverage=LEVERAGE,
    time_limit=TIME_LIMIT,
    bb_lengths=BOLLINGER_LENGTHS,
    bb_stds=BOLLINGER_STDS,
)
for config in strategy_configs:
    dump_dict_to_yaml("configs/", config)